# RFDiffusion3 Tutorial

This notebook walks through one example of using the **RFDiffusion3 ProtFlow runner**.

| # | Example | Key concept |
|---|---------|-------------|
| 1 | De novo design | Running RFD3 without any input structure |



## Imports and shared setup

We import ProtFlow, initialise the jobstarter, and create the runner.
Make sure you execute commands in an active ProtFlow environment.

In [ ]:
import logging
import os
import time

import pandas as pd

import protflow
from protflow.jobstarters import SbatchArrayJobstarter
from protflow.poses import Poses
from protflow.tools.rfdiffusion3 import RFdiffusion3
from protflow.residues import residue_selection

# logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)

# SbatchArrayJobstarter runs jobs by submitting a SLURM job
gpu_jobstarter = SbatchArrayJobstarter(max_cores=10, gpus=1, options="--time=00:10:00 ")

# single runner instance reused across all examples
rfdiffusion3 = RFdiffusion3(jobstarter=gpu_jobstarter)

# paths (relative to notebook location)
OUT_ROOT  = 'rfdiffusion3_data/outputs/'
os.makedirs(OUT_ROOT, exist_ok=True)

print('Setup complete.')
print(f'  Output root: {os.path.abspath(OUT_ROOT)}')

---
## Example 1 — De novo design

In **de novo** mode we provide no input structure. RFD3 generates proteins from scratch guided only by a length range.

### How it works

- We create an **empty `Poses` object** by passing `poses=None`.
- The runner detects `len(poses) == 0` and enters de novo mode.
- It builds a minimal input JSON containing only the `length` field.
- `settings_group_name` defaults automatically to `"denovo"` — no need to set it.
- After the run, `poses.df` is populated directly from the collected output scores.

### Input JSON written internally by the runner
```json
{
    "denovo": {
        "length": "100-100"
    }
}
```

In [ ]:
OUT_DENOVO = os.path.join(OUT_ROOT, 'rfd3_denovo')
os.makedirs(OUT_DENOVO, exist_ok=True)

# poses=None signals that there is no input structure
poses_denovo = Poses(
    poses=None,
    work_dir=OUT_DENOVO,
    jobstarter=gpu_jobstarter,
)

print(f'Empty poses created: {len(poses_denovo.df.index)} rows (expected 0)')

In [ ]:
# length='100' means exactly 100 residues
# 1 batch x diffusion_batch_size=8  ->  8 output structures
# overwrite=True ensures a clean run even if outputs already exist
poses_denovo = rfdiffusion3.run(
    poses=poses_denovo,
    prefix='rfdiffusion3',
    length='100',
    n_batches=1,
    diffusion_batch_size=8,
    overwrite=True,
)

print(f'De novo run complete: {len(poses_denovo.df.index)} output poses')

In [ ]:
# The runner populates poses.df with one row per output structure.
# Key columns:
#   description   unique name derived from the output filename
#   location      absolute path to the decompressed .cif file
#   metrics_*     scores from the RFD3 sidecar JSON

print('Columns in poses.df:')
print(list(poses_denovo.df.columns))

poses_denovo.df[['description', 'location']]

In [ ]:
scores_path = os.path.join(OUT_DENOVO, 'scores_denovo.json')
poses_denovo.save_scores(scores_path)
print(f'Scores saved to: {scores_path}')

---
## Summary

| Example | `poses=` | `overwrite=` | Key parameter | Output |
|---------|----------|-------------|---------------|--------|
| 1 — De novo | `None` | `True` | `length='100-100'` | 8 structures, no input reference |

### Next steps

- Make a motif conditioned design by defining an input .pdb file and unindexed residues.
- Increase `diffusion_batch_size` or `n_batches` to generate more designs.
- Use `multiplex_poses=N` to saturate multiple GPUs from a single input.
- Pipe `poses` into a ProteinMPNN or ESMFold runner for a full design loop.
- Use `fail_on_missing_output_poses=True` in production to catch silent RFD3 crashes.